# FinQA 数据资产准备

本 Notebook 为银行金融大模型量化项目准备三类资产：

- `calibration`：用于 AWQ/GPTQ/SmoothQuant 校准；
- `quality_dev` 和 `quality_test`：用于质量评估；
- `high_risk_regression`：固定的金融数值回归集。

数据会缓存到 Google Drive，避免 Colab 运行时重置后重复下载。

In [ ]:
!pip -q install -U datasets huggingface_hub pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 载入项目源码

如果你已经把项目目录上传到 Colab，请修改 `PROJECT_DIR`。如果项目放在 Git 仓库，可以填写 `REPO_URL` 自动克隆。

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = '/content/drive/MyDrive/OptimizationSystem'
REPO_URL = ''

if REPO_URL and not Path(PROJECT_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} 不存在，请上传项目或填写 REPO_URL。'
    )

sys.path.insert(0, PROJECT_DIR)

In [ ]:
import importlib
import src.data.finqa_assets as finqa_assets

importlib.reload(finqa_assets)
prepare_assets = finqa_assets.prepare_assets
print('FinQA loader:', finqa_assets.__file__)

OUTPUT_DIR = (
    '/content/drive/MyDrive/'
    'banking_llm_project/datasets'
)

assets = prepare_assets(
    output_dir=OUTPUT_DIR,
    calibration_size=1024,
    seed=42,
)

assets.summary()

FinQA loader: /content/drive/MyDrive/OptimizationSystem/src/data/finqa_assets.py


Saving the dataset (0/1 shards):   0%|          | 0/1024 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/883 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1064 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/91 [00:00<?, ? examples/s]

{'calibration': 1024,
 'quality_dev': 883,
 'quality_test': 1064,
 'high_risk_regression': 91,
 'manifest': {'finqa_dataset': 'czyssrs/FinQA',
  'verified_dataset': 'Aiera/finqa-verified',
  'finqa_revision': 'main',
  'verified_revision': 'default',
  'seed': 42,
  'calibration_size': 1024,
  'sizes': {'calibration': 1024,
   'quality_dev': 883,
   'quality_test': 1064,
   'high_risk_regression': 91},
  'schema_version': 1,
  'assets': {'calibration': {'path': 'calibration_finqa_1024',
    'version': 'main',
    'size': 1024,
    'sha256': '89dbd2dcd40a790cc5cb94a3728de2f9b0c070ab9b8a7f9eeae42c3ced851e9a',
    'required_fields': ['pre_text', 'post_text', 'table', 'qa.question']},
   'evaluation_dev': {'path': 'quality_dev_finqa',
    'version': 'main',
    'size': 883,
    'sha256': '7ba8d507634daa266268414448444421a18868e926632d805949058118b8a6d8',
    'required_fields': ['pre_text', 'post_text', 'table', 'qa.question']},
   'evaluation_test': {'path': 'quality_test_finqa',
    'vers

In [ ]:
print('校准集:', len(assets.calibration))
print('质量调优集:', len(assets.quality_dev))
print('最终质量评估集:', len(assets.quality_test))
print('高风险回归集:', len(assets.high_risk_regression))
print('第一条校准 Prompt:')
print(assets.calibration_prompts[0][:2000])

校准集: 1024
质量调优集: 883
最终质量评估集: 1064
高风险回归集: 91
第一条校准 Prompt:
You are a financial analysis assistant.

Financial context:
contractual cash flows following is a summary of our contractual payment obligations related to our consolidated debt , contingent consideration , operating leases , other commitments and long-term liabilities at september 30 , 2011 ( see notes 9 and 13 to the consolidated financial statements contained this annual report ) , ( in thousands ) : .

Financial table:
obligation | payments due by period total | payments due by period less than 1year | payments due by period 1-3 years | payments due by period 3-5 years | payments due by period thereafter
short-term debt obligations | $ 26677 | $ 26677 | $ 2014 | $ 2014 | $ 2014
cash premium on convertible notes due march 2012 ( 1 ) | 23558 | 23558 | 2014 | 2014 | 2014
other commitments ( 2 ) | 5170 | 3398 | 1772 | 2014 | 2014
operating lease obligations | 37788 | 8247 | 13819 | 9780 | 5942
contingent consideration for busi

## 接入量化算法

`assets.calibration_prompts` 可以直接传给你的 AWQ/GPTQ 校准流程。不要把答案字段拼进校准 Prompt。

In [ ]:
calibration_prompts = assets.calibration_prompts

for prompt in calibration_prompts[:3]:
    print(prompt[:500])
    print('---')

You are a financial analysis assistant.

Financial context:
contractual cash flows following is a summary of our contractual payment obligations related to our consolidated debt , contingent consideration , operating leases , other commitments and long-term liabilities at september 30 , 2011 ( see notes 9 and 13 to the consolidated financial statements contained this annual report ) , ( in thousands ) : .

Financial table:
obligation | payments due by period total | payments due by period less t
---
You are a financial analysis assistant.

Financial context:
capital asset purchases associated with the retail segment were $ 294 million in 2007 , bringing the total capital asset purchases since inception of the retail segment to $ 1.0 billion .
as of september 29 , 2007 , the retail segment had approximately 7900 employees and had outstanding operating lease commitments associated with retail store space and related facilities of $ 1.1 billion .
the company would incur substantial costs 

In [ ]:
from src.data.finqa_assets import load_assets

reloaded_assets = load_assets(OUTPUT_DIR)
reloaded_assets.summary()

{'calibration': 1024,
 'quality_dev': 883,
 'quality_test': 1064,
 'high_risk_regression': 91,
 'manifest': {'finqa_dataset': 'czyssrs/FinQA',
  'verified_dataset': 'Aiera/finqa-verified',
  'finqa_revision': 'main',
  'verified_revision': 'default',
  'seed': 42,
  'calibration_size': 1024,
  'sizes': {'calibration': 1024,
   'quality_dev': 883,
   'quality_test': 1064,
   'high_risk_regression': 91},
  'schema_version': 1,
  'assets': {'calibration': {'path': 'calibration_finqa_1024',
    'version': 'main',
    'size': 1024,
    'sha256': '89dbd2dcd40a790cc5cb94a3728de2f9b0c070ab9b8a7f9eeae42c3ced851e9a',
    'required_fields': ['pre_text', 'post_text', 'table', 'qa.question']},
   'evaluation_dev': {'path': 'quality_dev_finqa',
    'version': 'main',
    'size': 883,
    'sha256': '7ba8d507634daa266268414448444421a18868e926632d805949058118b8a6d8',
    'required_fields': ['pre_text', 'post_text', 'table', 'qa.question']},
   'evaluation_test': {'path': 'quality_test_finqa',
    'vers

## 使用规则

1. 用 `calibration` 计算量化参数。
2. 用 `quality_dev` 选择 group size、敏感层和混合精度。
3. 配置冻结后才使用 `quality_test` 生成最终报告。
4. 每个 FP16、INT8、INT4、剪枝或后端版本都运行 `high_risk_regression`。
5. 重点统计“FP16 正确但压缩模型错误”的新增退化。

## TODO 4.1：模型输入校验（Colab）

下面先生成一个合成微型 GPT-2，验证 manifest、文件检查、SHA-256、FP16 加载和 Transformers 推理的完整链路。运行前请在 Colab 选择 **运行时 → 更改运行时类型 → T4 GPU**。

> 合成模型只用于验证校验工具，不能据此勾选真实项目的 Todo 4.1。正式验收时，请按照 `docs/MODEL_INPUTS.md` 将 manifest 指向上游交付的真实模型。

In [ ]:
%pip -q install -U transformers accelerate safetensors tokenizers

In [ ]:
import json
from pathlib import Path

import torch
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast

if not torch.cuda.is_available():
    raise RuntimeError('请先把 Colab 运行时切换为 T4 GPU，再重新运行此单元格。')

TEST_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_test')
MODEL_DIR = TEST_ROOT / 'synthetic_tiny_gpt2'
REPORT_DIR = Path('/content/drive/MyDrive/OptimizationSystem/out/model_input_test')
MANIFEST_PATH = TEST_ROOT / 'model_manifest.synthetic.json'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

vocabulary = {
    '[PAD]': 0, '[UNK]': 1, '[BOS]': 2, '[EOS]': 3,
    'hello': 4, 'bank': 5, 'finance': 6, 'test': 7,
}
tokenizer_backend = Tokenizer(WordLevel(vocabulary, unk_token='[UNK]'))
tokenizer_backend.pre_tokenizer = Whitespace()
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_backend,
    unk_token='[UNK]', pad_token='[PAD]',
    bos_token='[BOS]', eos_token='[EOS]',
)
tokenizer.save_pretrained(MODEL_DIR)

config = GPT2Config(
    vocab_size=len(vocabulary), n_positions=32, n_ctx=32,
    n_embd=32, n_layer=2, n_head=4,
    bos_token_id=2, eos_token_id=3, pad_token_id=0,
)
GPT2LMHeadModel(config).save_pretrained(MODEL_DIR, safe_serialization=True)
(MODEL_DIR / 'LICENSE_TEST_ONLY.txt').write_text(
    'Synthetic model generated locally for internal validator testing only.\n',
    encoding='utf-8',
)

weight_files = sorted(path.name for path in MODEL_DIR.glob('*.safetensors'))
tokenizer_files = [
    name for name in ('tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json')
    if (MODEL_DIR / name).is_file()
]
manifest = {
    'schema_version': 1,
    'model_id': 'local/synthetic-tiny-gpt2',
    'model_path': str(MODEL_DIR),
    'version': 'test-1.0',
    'commit': 'generated-locally',
    'weight_files': weight_files,
    'config_file': 'config.json',
    'tokenizer_files': tokenizer_files,
    'expected_sha256': {},
    'license': {
        'name': 'Synthetic test fixture',
        'file': 'LICENSE_TEST_ONLY.txt',
        'internal_use_approved': True,
        'usage_scope': 'Validator smoke testing only',
    },
    'runtime': {
        'framework': 'transformers',
        'model_class': 'causal_lm',
        'dtype': 'float16',
        'device': 'cuda:0',
        'trust_remote_code': False,
        'smoke_test_prompt': 'hello bank',
    },
}
MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('模型目录:', MODEL_DIR)
print('Manifest:', MANIFEST_PATH)
print('权重文件:', weight_files)
print('Tokenizer 文件:', tokenizer_files)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

模型目录: /content/drive/MyDrive/banking_llm_project/model_input_test/synthetic_tiny_gpt2
Manifest: /content/drive/MyDrive/banking_llm_project/model_input_test/model_manifest.synthetic.json
权重文件: ['model.safetensors']
Tokenizer 文件: ['tokenizer.json', 'tokenizer_config.json']


In [ ]:
import importlib
import src.input_validation.model_input as model_input_validation

importlib.reload(model_input_validation)
static_report = model_input_validation.validate_model_input(MANIFEST_PATH)
static_payload = static_report.to_dict()
(REPORT_DIR / 'static.json').write_text(
    json.dumps(static_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(static_payload, ensure_ascii=False, indent=2))
assert static_report.validation_passed
assert not static_report.complete
print('静态检查通过；两项运行时检查尚未执行。')

{
  "manifest": "/content/drive/MyDrive/banking_llm_project/model_input_test/model_manifest.synthetic.json",
  "model_id": "local/synthetic-tiny-gpt2",
  "validation_passed": true,
  "complete": false,
  "checks": [
    {
      "name": "base_model_weights_received",
      "status": "pass",
      "detail": "Validated 1 non-empty weight file(s)."
    },
    {
      "name": "config_and_tokenizer_received",
      "status": "pass",
      "detail": "Config and tokenizer files are present and non-empty."
    },
    {
      "name": "license_and_internal_scope_confirmed",
      "status": "pass",
      "detail": "License file, approval, and internal usage scope are recorded."
    },
    {
      "name": "version_commit_and_weight_hash_recorded",
      "status": "pass",
      "detail": "Recorded identity and 1 SHA-256 hash(es)."
    },
    {
      "name": "fp16_or_bf16_model_load",
      "status": "not_run",
      "detail": "Run with --runtime-check to load the model."
    },
    {
      "name": "

In [ ]:
runtime_report = model_input_validation.validate_model_input(
    MANIFEST_PATH, runtime_check=True
)
runtime_payload = runtime_report.to_dict()
(REPORT_DIR / 'runtime.json').write_text(
    json.dumps(runtime_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(runtime_payload, ensure_ascii=False, indent=2))
assert runtime_report.complete, '存在未通过项，请查看上面的 detail。'
print('TODO 4.1 校验工具链冒烟测试：PASS')
print('报告目录:', REPORT_DIR)

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

{
  "manifest": "/content/drive/MyDrive/banking_llm_project/model_input_test/model_manifest.synthetic.json",
  "model_id": "local/synthetic-tiny-gpt2",
  "validation_passed": true,
  "complete": true,
  "checks": [
    {
      "name": "base_model_weights_received",
      "status": "pass",
      "detail": "Validated 1 non-empty weight file(s)."
    },
    {
      "name": "config_and_tokenizer_received",
      "status": "pass",
      "detail": "Config and tokenizer files are present and non-empty."
    },
    {
      "name": "license_and_internal_scope_confirmed",
      "status": "pass",
      "detail": "License file, approval, and internal usage scope are recorded."
    },
    {
      "name": "version_commit_and_weight_hash_recorded",
      "status": "pass",
      "detail": "Recorded identity and 1 SHA-256 hash(es)."
    },
    {
      "name": "fp16_or_bf16_model_load",
      "status": "pass",
      "detail": "Loaded float16 model on cuda:0."
    },
    {
      "name": "target_framework

### 使用真实模型验收

合成测试通过后，复制 `inputs/model_manifest.example.json`，将 `model_path`、权重分片、版本、commit、许可证和运行时配置替换为上游真实信息。许可证负责人确认前，不要把 `internal_use_approved` 设置为 `true`。真实模型生成的 `runtime.json` 六项全部为 `pass` 后，才能勾选 Todo 4.1。

## TODO 4.1：真实基础模型验收

本节默认使用 `Qwen/Qwen2.5-0.5B-Instruct` 完成真实模型验收。模型约 1 GB，可在 T4 GPU 上以 FP16 运行。代码会固定 Hugging Face commit、下载模型到 Google Drive、计算权重哈希并生成静态与运行时报告。

运行配置单元格前，请阅读模型卡和仓库中的许可证。只有你确认许可证允许当前内部使用范围后，才能把 `INTERNAL_USE_APPROVED` 改成 `True`。

In [ ]:
REAL_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
REAL_MODEL_REVISION = 'main'
INTERNAL_USE_APPROVED = True  # 阅读并确认许可证后手动改为 True
INTERNAL_USAGE_SCOPE = 'Internal research, evaluation, and model optimization only'
REAL_TEST_PROMPT = '请用一句话说明什么是贷款利率。'

print('模型卡: https://huggingface.co/' + REAL_MODEL_ID)
print('内部使用许可已确认:', INTERNAL_USE_APPROVED)

模型卡: https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct
内部使用许可已确认: True


In [ ]:
from huggingface_hub import HfApi, snapshot_download

REAL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_real')
REAL_MODEL_DIR = REAL_ROOT / 'qwen2.5-0.5b-instruct'
REAL_REPORT_DIR = Path('/content/drive/MyDrive/OptimizationSystem/out/model_input_real')
REAL_MANIFEST_PATH = REAL_ROOT / 'model_manifest.json'
REAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
REAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)

model_info = HfApi().model_info(REAL_MODEL_ID, revision=REAL_MODEL_REVISION)
REAL_MODEL_COMMIT = model_info.sha
reported_license = getattr(model_info.card_data, 'license', None) or 'not-declared'
print('解析后的 commit:', REAL_MODEL_COMMIT)
print('模型卡声明的许可证:', reported_license)

snapshot_download(
    repo_id=REAL_MODEL_ID,
    revision=REAL_MODEL_COMMIT,
    local_dir=REAL_MODEL_DIR,
    allow_patterns=[
        '*.json', '*.safetensors', '*.model', '*.tiktoken',
        'vocab.*', 'merges.*', 'LICENSE*',
    ],
)
print('真实模型目录:', REAL_MODEL_DIR)

解析后的 commit: 7ae557604adf67be50417f59c2c2f167def9a775
模型卡声明的许可证: apache-2.0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

真实模型目录: /content/drive/MyDrive/banking_llm_project/model_input_real/qwen2.5-0.5b-instruct


In [ ]:
weight_files = sorted(
    str(path.relative_to(REAL_MODEL_DIR))
    for pattern in ('*.safetensors', 'pytorch_model*.bin')
    for path in REAL_MODEL_DIR.rglob(pattern)
    if path.is_file()
)
tokenizer_names = (
    'tokenizer.json', 'tokenizer_config.json', 'tokenizer.model',
    'vocab.json', 'merges.txt', 'special_tokens_map.json',
)
tokenizer_files = [
    name for name in tokenizer_names if (REAL_MODEL_DIR / name).is_file()
]
license_files = sorted(
    path.name for path in REAL_MODEL_DIR.glob('LICENSE*') if path.is_file()
)
if not license_files:
    raise FileNotFoundError('模型仓库没有下载到 LICENSE 文件，不能完成许可证检查。')

real_manifest = {
    'schema_version': 1,
    'model_id': REAL_MODEL_ID,
    'model_path': str(REAL_MODEL_DIR),
    'version': 'Qwen2.5-0.5B-Instruct',
    'commit': REAL_MODEL_COMMIT,
    'weight_files': weight_files,
    'config_file': 'config.json',
    'tokenizer_files': tokenizer_files,
    'expected_sha256': {},
    'license': {
        'name': reported_license,
        'file': license_files[0],
        'internal_use_approved': INTERNAL_USE_APPROVED,
        'usage_scope': INTERNAL_USAGE_SCOPE,
    },
    'runtime': {
        'framework': 'transformers',
        'model_class': 'causal_lm',
        'dtype': 'float16',
        'device': 'cuda:0',
        'trust_remote_code': False,
        'smoke_test_prompt': REAL_TEST_PROMPT,
    },
}
REAL_MANIFEST_PATH.write_text(
    json.dumps(real_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Manifest:', REAL_MANIFEST_PATH)
print('权重文件:', weight_files)
print('Tokenizer 文件:', tokenizer_files)
print('许可证文件:', license_files[0])

Manifest: /content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json
权重文件: ['model.safetensors']
Tokenizer 文件: ['tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt']
许可证文件: LICENSE


In [ ]:
real_static_report = model_input_validation.validate_model_input(
    REAL_MANIFEST_PATH
)
real_static_payload = real_static_report.to_dict()
(REAL_REPORT_DIR / 'static.json').write_text(
    json.dumps(real_static_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(real_static_payload, ensure_ascii=False, indent=2))
assert real_static_report.validation_passed, (
    '静态检查未通过。若仅许可证项失败，请确认许可证后把 '
    'INTERNAL_USE_APPROVED 改为 True，并重新运行配置、manifest 和本单元格。'
)

{
  "manifest": "/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "validation_passed": true,
  "complete": false,
  "checks": [
    {
      "name": "base_model_weights_received",
      "status": "pass",
      "detail": "Validated 1 non-empty weight file(s)."
    },
    {
      "name": "config_and_tokenizer_received",
      "status": "pass",
      "detail": "Config and tokenizer files are present and non-empty."
    },
    {
      "name": "license_and_internal_scope_confirmed",
      "status": "pass",
      "detail": "License file, approval, and internal usage scope are recorded."
    },
    {
      "name": "version_commit_and_weight_hash_recorded",
      "status": "pass",
      "detail": "Recorded identity and 1 SHA-256 hash(es)."
    },
    {
      "name": "fp16_or_bf16_model_load",
      "status": "not_run",
      "detail": "Run with --runtime-check to load the model."
    },
    {
      "name": "target_fr

In [ ]:
import gc

if not torch.cuda.is_available():
    raise RuntimeError('请先把 Colab 运行时切换为 T4 GPU。')
gc.collect()
torch.cuda.empty_cache()

real_runtime_report = model_input_validation.validate_model_input(
    REAL_MANIFEST_PATH, runtime_check=True
)
real_runtime_payload = real_runtime_report.to_dict()
(REAL_REPORT_DIR / 'runtime.json').write_text(
    json.dumps(real_runtime_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(real_runtime_payload, ensure_ascii=False, indent=2))
assert real_runtime_report.complete, '真实模型存在未通过项，请查看上面的 detail。'
print('TODO 4.1 真实模型验收：PASS')
print('报告目录:', REAL_REPORT_DIR)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{
  "manifest": "/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "validation_passed": true,
  "complete": true,
  "checks": [
    {
      "name": "base_model_weights_received",
      "status": "pass",
      "detail": "Validated 1 non-empty weight file(s)."
    },
    {
      "name": "config_and_tokenizer_received",
      "status": "pass",
      "detail": "Config and tokenizer files are present and non-empty."
    },
    {
      "name": "license_and_internal_scope_confirmed",
      "status": "pass",
      "detail": "License file, approval, and internal usage scope are recorded."
    },
    {
      "name": "version_commit_and_weight_hash_recorded",
      "status": "pass",
      "detail": "Recorded identity and 1 SHA-256 hash(es)."
    },
    {
      "name": "fp16_or_bf16_model_load",
      "status": "pass",
      "detail": "Loaded float16 model on cuda:0."
    },
    {
      "name": "target_framework_inferenc

### 完成条件

最后一个单元格输出 `TODO 4.1 真实模型验收：PASS`，且 `runtime.json` 中 `complete` 为 `true` 后，将 `model_manifest.json`、`static.json` 和 `runtime.json` 拉回项目，即可据此更新 Todo 4.1。

## TODO 4.2：评测资产输入验收

本节重新生成增强版数据 manifest，实际加载校准集、开发/最终评估集和高风险回归集，检查路径、版本、隔离、字段、指标接口、关键错误规则、样本数与 SHA-256。数据仍保存在原 `banking_llm_project/datasets` 路径，只有验收结果写入 `OptimizationSystem/out/dataset_input`。

In [56]:
import importlib

importlib.reload(finqa_assets)
prepare_assets = finqa_assets.prepare_assets
assets = prepare_assets(
    output_dir=OUTPUT_DIR,
    calibration_size=1024,
    seed=42,
)
DATASET_MANIFEST_PATH = Path(OUTPUT_DIR) / 'manifest.json'
print('增强版数据 Manifest:', DATASET_MANIFEST_PATH)
print(json.dumps(assets.manifest, ensure_ascii=False, indent=2))

Filter:   0%|          | 0/6251 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1024 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/883 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1064 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/91 [00:00<?, ? examples/s]

增强版数据 Manifest: /content/drive/MyDrive/banking_llm_project/datasets/manifest.json
{
  "finqa_dataset": "czyssrs/FinQA",
  "verified_dataset": "Aiera/finqa-verified",
  "finqa_revision": "main",
  "verified_revision": "default",
  "seed": 42,
  "calibration_size": 1024,
  "sizes": {
    "calibration": 1024,
    "quality_dev": 883,
    "quality_test": 1064,
    "high_risk_regression": 91
  },
  "schema_version": 1,
  "assets": {
    "calibration": {
      "path": "calibration_finqa_1024",
      "version": "main",
      "size": 1024,
      "sha256": "34af0e443492d4fdc831712c67cd6f12168e923b187b0175817ed9457bd6adfd",
      "required_fields": [
        "pre_text",
        "post_text",
        "table",
        "qa.question"
      ]
    },
    "evaluation_dev": {
      "path": "quality_dev_finqa",
      "version": "main",
      "size": 883,
      "sha256": "7ba8d507634daa266268414448444421a18868e926632d805949058118b8a6d8",
      "required_fields": [
        "pre_text",
        "post_text",
  

In [57]:
import src.evaluation.finqa_metrics as finqa_metrics

importlib.reload(finqa_metrics)
metric_smoke = finqa_metrics.evaluate_numeric_answers(
    predictions=['$100', '12.5%'],
    references=['100', '0.125'],
)
regression_smoke = finqa_metrics.classify_numeric_regression(
    baseline_prediction='100',
    candidate_prediction='99',
    reference='100',
)
print('数值指标冒烟结果:', metric_smoke)
print('新增退化判定:', regression_smoke)
assert metric_smoke['numeric_accuracy'] == 1.0
assert regression_smoke['new_regression'] is True

数值指标冒烟结果: {'total': 2, 'parsed': 2, 'correct': 2, 'parse_rate': 1.0, 'numeric_accuracy': 1.0}
新增退化判定: {'baseline_correct': True, 'candidate_correct': False, 'new_regression': True, 'critical_error_type': 'numeric_answer_mismatch'}


In [58]:
import sys
import importlib

PROJECT_DIR = "/content/drive/MyDrive/OptimizationSystem"
sys.path.insert(0, PROJECT_DIR)
importlib.invalidate_caches()

import src.evaluation.finqa_metrics as finqa_metrics
import src.input_validation.dataset_input asdataset_input_validation

print(finqa_metrics.__file__)
print(dataset_input_validation.__file__)

SyntaxError: invalid syntax (3144036247.py, line 9)

In [59]:
import shutil
import src.input_validation.dataset_input as dataset_input_validation

importlib.reload(dataset_input_validation)
dataset_report = dataset_input_validation.validate_dataset_input(
    DATASET_MANIFEST_PATH
)
dataset_payload = dataset_report.to_dict()
DATASET_RESULT_DIR = Path(
    '/content/drive/MyDrive/OptimizationSystem/out/dataset_input'
)
DATASET_RESULT_DIR.mkdir(parents=True, exist_ok=True)
(DATASET_RESULT_DIR / 'validation.json').write_text(
    json.dumps(dataset_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
shutil.copy2(
    DATASET_MANIFEST_PATH,
    DATASET_RESULT_DIR / 'dataset_manifest.json',
)
print(json.dumps(dataset_payload, ensure_ascii=False, indent=2))
assert dataset_report.complete, '评测资产存在未通过项，请查看上面的 detail。'
print('TODO 4.2 评测资产输入验收：PASS')
print('结果目录:', DATASET_RESULT_DIR)

{
  "manifest": "/content/drive/MyDrive/banking_llm_project/datasets/manifest.json",
  "complete": true,
  "checks": [
    {
      "name": "dataset_manifest_received",
      "status": "pass",
      "detail": "Dataset manifest schema and four asset roles are present."
    },
    {
      "name": "calibration_path_and_version_received",
      "status": "pass",
      "detail": "Calibration asset is loadable and versioned."
    },
    {
      "name": "evaluation_paths_and_versions_received",
      "status": "pass",
      "detail": "Development and final evaluation assets are loadable and versioned."
    },
    {
      "name": "regression_path_and_version_received",
      "status": "pass",
      "detail": "High-risk regression asset is loadable and versioned."
    },
    {
      "name": "calibration_and_evaluation_isolated",
      "status": "pass",
      "detail": "Calibration/evaluation overlap: 0; regression/final-test overlap: 0."
    },
    {
      "name": "sample_schema_and_fields_confi

### 4.2 完成条件

看到 `TODO 4.2 评测资产输入验收：PASS` 后，将 `out/dataset_input/dataset_manifest.json` 和 `out/dataset_input/validation.json` 拉回项目。确认 `validation.json` 中九项均为 `pass` 且 `complete` 为 `true`，即可勾选 Todo 4.2。

## TODO 4.3：硬件环境输入验收

本节采集 Colab 的真实 NVIDIA GPU、显存、驱动、CUDA、容器、PyTorch、Transformers 和拓扑信息。硬件 manifest 保存在原资产目录 `banking_llm_project/hardware`，项目结果保存在 `OptimizationSystem/out/hardware`。

> Colab 没有昇腾设备，因此本节只能完成 NVIDIA 部分。Todo 4.3 必须等真实昇腾机器的 manifest 合并通过后才能整体勾选。

In [ ]:
import importlib
import src.input_validation.hardware_input as hardware_input_validation

importlib.reload(hardware_input_validation)
HARDWARE_ASSET_DIR = Path(
    '/content/drive/MyDrive/banking_llm_project/hardware'
)
HARDWARE_RESULT_DIR = Path(
    '/content/drive/MyDrive/OptimizationSystem/out/hardware'
)
NVIDIA_MANIFEST_PATH = HARDWARE_ASSET_DIR / 'nvidia_colab.json'
HARDWARE_ASSET_DIR.mkdir(parents=True, exist_ok=True)
HARDWARE_RESULT_DIR.mkdir(parents=True, exist_ok=True)

nvidia_manifest = hardware_input_validation.collect_nvidia_environment()
NVIDIA_MANIFEST_PATH.write_text(
    json.dumps(nvidia_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(nvidia_manifest, ensure_ascii=False, indent=2))
assert nvidia_manifest['accelerator']['available'], '没有检测到 NVIDIA GPU。'
print('NVIDIA 环境采集：PASS')
print('硬件 Manifest:', NVIDIA_MANIFEST_PATH)

In [ ]:
import shutil

nvidia_only_report = hardware_input_validation.validate_hardware_inputs(
    NVIDIA_MANIFEST_PATH
)
nvidia_only_payload = nvidia_only_report.to_dict()
(HARDWARE_RESULT_DIR / 'nvidia_validation.json').write_text(
    json.dumps(nvidia_only_payload, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
shutil.copy2(
    NVIDIA_MANIFEST_PATH,
    HARDWARE_RESULT_DIR / 'nvidia_colab.json',
)
print(json.dumps(nvidia_only_payload, ensure_ascii=False, indent=2))
assert nvidia_only_payload['checks'][0]['status'] == 'pass'
assert nvidia_only_payload['checks'][1]['status'] == 'pass'
assert nvidia_only_report.complete is False
print('NVIDIA 两项验收：PASS；昇腾与双平台合并验收：PENDING')
print('结果目录:', HARDWARE_RESULT_DIR)

### 昇腾环境下一步

在真实昇腾机器同步项目代码后执行：

```bash
python -m src.input_validation.hardware_input collect \
  --target ascend \
  --output /path/to/hardware/ascend.json
```

根据 `npu-smi` 和实际安装补齐芯片型号、数量、显存、CANN、驱动、固件、推理框架及通信字段，再与 `nvidia_colab.json` 执行合并验收。七项全部为 `pass` 后才能完成 Todo 4.3。